# 01.8 Saving, Loading, and Inference

After training a model, the practical questions are how to save the learned parameters, how to recreate the model later, and how to run predictions safely. This notebook turns a model from a temporary in-memory object into something you can reuse.

The important distinction is that a checkpoint does not magically contain your Python class definition. To load parameters correctly, you must recreate the same architecture and then load the saved `state_dict` into it.

## Learning Goals

After this notebook, you should be able to:

1. Understand what a `state_dict` is.
2. Save and load model parameters.
3. Save and load a training checkpoint.
4. Switch correctly to `eval()` before inference.
5. Use `torch.no_grad()` during inference.
6. Run predictions on new samples.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
import torch.nn as nn

## Prepare a Small Model

We first define a simple model and train it for a few steps so its parameters are no longer purely random.


In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self, in_features=2, hidden_features=8, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, num_classes),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(1)
model = SmallClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

X = torch.tensor(
    [
        [-1.0, -1.2],
        [-0.8, -0.5],
        [1.1, 0.9],
        [0.9, 1.2],
    ],
    dtype=torch.float32,
)
y = torch.tensor([0, 0, 1, 1], dtype=torch.long)

for _ in range(20):
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

print("training loss =", loss.item())

## 2. `state_dict`

`state_dict` is the most common way to save model parameters.

It is essentially a dictionary that stores parameter tensors.


In [ ]:
state = model.state_dict()
print(type(state))
print(list(state.keys()))

## Save and Load Model Parameters

We use a temporary directory here so that validation does not leave files behind in the repository.


In [ ]:
with TemporaryDirectory() as tmp_dir:
    save_path = Path(tmp_dir) / "model_state.pt"
    torch.save(model.state_dict(), save_path)

    loaded_model = SmallClassifier()
    loaded_model.load_state_dict(torch.load(save_path))

    sample = torch.tensor([[0.8, 1.0]], dtype=torch.float32)
    out1 = model(sample)
    out2 = loaded_model(sample)

    print("save_path =", save_path)
    print("original output =", out1)
    print("loaded output =", out2)
    print("allclose =", torch.allclose(out1, out2))

The important point is that `state_dict` saves parameter values, not the model architecture. When loading, you must first create a model object with the same layer structure. Then `load_state_dict` copies the saved parameter values into that object.

## Saving a Training Checkpoint

Saving only `model.state_dict()` is enough when you only need inference. To resume training, you also need training state. That usually includes the optimizer state, current epoch or step, scheduler state if one exists, and the configuration used for the run.

Without optimizer state, resumed training may not behave like a true continuation because optimizers such as Adam keep internal running statistics.

In [ ]:
with TemporaryDirectory() as tmp_dir:
    ckpt_path = Path(tmp_dir) / "checkpoint.pt"

    checkpoint = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": 20,
        "note": "demo checkpoint",
    }
    torch.save(checkpoint, ckpt_path)

    loaded_ckpt = torch.load(ckpt_path)

    resume_model = SmallClassifier()
    resume_optimizer = torch.optim.Adam(resume_model.parameters(), lr=0.05)
    resume_model.load_state_dict(loaded_ckpt["model_state"])
    resume_optimizer.load_state_dict(loaded_ckpt["optimizer_state"])

    print("loaded epoch =", loaded_ckpt["epoch"])
    print("loaded note =", loaded_ckpt["note"])

## Inference

Two key habits for inference:

1. `model.eval()`
2. `with torch.no_grad():`

Both of these steps matter.


In [ ]:
model.eval()

new_x = torch.tensor(
    [
        [-1.1, -0.9],
        [1.2, 1.0],
        [0.2, 0.1],
    ],
    dtype=torch.float32,
)

with torch.no_grad():
    logits = model(new_x)
    probs = torch.softmax(logits, dim=1)
    preds = logits.argmax(dim=1)

print("logits =\n", logits)
print("probs =\n", probs)
print("preds =", preds)

The inference code separates three ideas. Logits are the raw scores produced by the model. Softmax turns those scores into probabilities that sum to 1 across classes. The final predicted class is the index of the largest probability. Keeping these concepts separate makes debugging classification outputs much easier.

## A Simple Prediction Function

For reuse, it is common to wrap inference logic into a function.


In [ ]:
def predict_classes(model, x):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        preds = logits.argmax(dim=1)
    return preds


preds = predict_classes(model, new_x)
print(preds)

In [ ]:
# Exercise 1
#
# Answer in one or two full sentences:
# Why do we usually call model.eval() before inference?
#
# Your answer should mention layers such as Dropout or BatchNorm and the fact
# that inference should use evaluation behavior, not training behavior.

Exercise 1 Reference Answer

We call `model.eval()` before inference so layers like Dropout and BatchNorm use inference behavior instead of training behavior.

In [ ]:
# Exercise 2
#
# Implement predict_proba(model, x).
#
# Requirements:
# - Put the model in eval mode.
# - Use torch.no_grad() so PyTorch does not build a gradient graph.
# - Run the model on x to get logits.
# - Return softmax probabilities across dim=1.

def predict_proba(model, x):
    # TODO
    pass


# print(predict_proba(model, new_x))

In [ ]:
# Exercise 2 Reference Solution

def predict_proba_solution(model, x):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
    return probs


print(predict_proba_solution(model, new_x))

In [ ]:
# Exercise 3
#
# Explain why saving only model parameters is not enough to resume training.
#
# Write two or three full sentences. Your answer should mention optimizer state
# and at least one other piece of training metadata, such as epoch, scheduler
# state, config, or metric history.

Exercise 3 Reference Answer

Saving only model parameters is enough for inference, but not for resuming training. To resume training faithfully, you usually also need optimizer state, current epoch or step, scheduler state, config, and metric history.

## Summary

There are three most important takeaways in this notebook:

1. the most common way to save a model is saving its `state_dict`
2. inference usually uses `eval()` + `no_grad()`
3. resuming training usually needs a checkpoint, not just model parameters

You should now be able to answer:

1. Why is `state_dict` the common saving format?
2. Why must the model architecture match when loading?
3. Why should inference avoid gradient tracking?

Suggested next step:

- Move to the Phase 1 mini-project notebook and connect data prep, model building, training, and inference end to end.